In [34]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", None)

In [35]:
train.head()

,Employee ID,Age,Gender,Years at Company,Job Role,Monthly Income,Work-Life Balance,Job Satisfaction,Performance Rating,Number of Promotions,Overtime,Distance from Home,Education Level,Marital Status,Number of Dependents,Job Level,Company Size,Company Tenure,Remote Work,Leadership Opportunities,Innovation Opportunities,Company Reputation,Employee Recognition,Attrition,data_source
0,8410,31,Male,19,Education,5390,Excellent,Medium,Average,2,No,22,Associate Degree,Married,0,Mid,Medium,89,No,No,No,Excellent,Medium,Stayed,train
1,64756,59,Female,4,Media,5534,Poor,High,Low,3,No,21,Master’s Degree,Divorced,3,Mid,Medium,21,No,No,No,Fair,Low,Stayed,train
2,30257,24,Female,10,Healthcare,8159,Good,High,Low,0,No,11,Bachelor’s Degree,Married,3,Mid,Medium,74,No,No,No,Poor,Low,Stayed,train
3,65791,36,Female,7,Education,3989,Good,High,High,1,No,27,High School,Single,2,Mid,Small,50,Yes,No,No,Good,Medium,Stayed,train
4,65026,56,Male,41,Education,4821,Fair,Very High,Average,0,Yes,71,High School,Divorced,0,Senior,Medium,68,No,No,No,Fair,Medium,Stayed,train


In [36]:
print("Train columns:", list(train.columns))
print("Test columns:", list(test.columns))

Train columns: ['Employee ID', 'Age', 'Gender', 'Years at Company', 'Job Role', 'Monthly Income', 'Work-Life Balance', 'Job Satisfaction', 'Performance Rating', 'Number of Promotions', 'Overtime', 'Distance from Home', 'Education Level', 'Marital Status', 'Number of Dependents', 'Job Level', 'Company Size', 'Company Tenure', 'Remote Work', 'Leadership Opportunities', 'Innovation Opportunities', 'Company Reputation', 'Employee Recognition', 'Attrition', 'data_source']
Test columns: ['Employee ID', 'Age', 'Gender', 'Years at Company', 'Job Role', 'Monthly Income', 'Work-Life Balance', 'Job Satisfaction', 'Performance Rating', 'Number of Promotions', 'Overtime', 'Distance from Home', 'Education Level', 'Marital Status', 'Number of Dependents', 'Job Level', 'Company Size', 'Company Tenure', 'Remote Work', 'Leadership Opportunities', 'Innovation Opportunities', 'Company Reputation', 'Employee Recognition', 'Attrition', 'data_source']


In [37]:
overlap = set(train["Employee ID"]) & set(test["Employee ID"])
print("Overlapping Employee IDs:", len(overlap))

Overlapping Employee IDs: 0


In [38]:
train["data_source"] = "train"
test["data_source"] = "test"

df = pd.concat([train, test], axis=0, ignore_index=True)

print("Combined shape:", df.shape)
print("Total rows:", len(df))
df["data_source"].value_counts()

Combined shape: (74498, 25)
Total rows: 74498


data_source
train    59598
test     14900
Name: count, dtype: int64

In [39]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace("-", "_")
    .str.replace(" ", "_")
)

df.columns.tolist()

['employee_id',
 'age',
 'gender',
 'years_at_company',
 'job_role',
 'monthly_income',
 'work_life_balance',
 'job_satisfaction',
 'performance_rating',
 'number_of_promotions',
 'overtime',
 'distance_from_home',
 'education_level',
 'marital_status',
 'number_of_dependents',
 'job_level',
 'company_size',
 'company_tenure',
 'remote_work',
 'leadership_opportunities',
 'innovation_opportunities',
 'company_reputation',
 'employee_recognition',
 'attrition',
 'data_source']

In [40]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 74498 entries, 0 to 74497
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   employee_id               74498 non-null  int64
 1   age                       74498 non-null  int64
 2   gender                    74498 non-null  str  
 3   years_at_company          74498 non-null  int64
 4   job_role                  74498 non-null  str  
 5   monthly_income            74498 non-null  int64
 6   work_life_balance         74498 non-null  str  
 7   job_satisfaction          74498 non-null  str  
 8   performance_rating        74498 non-null  str  
 9   number_of_promotions      74498 non-null  int64
 10  overtime                  74498 non-null  str  
 11  distance_from_home        74498 non-null  int64
 12  education_level           74498 non-null  str  
 13  marital_status            74498 non-null  str  
 14  number_of_dependents      74498 non-null  int64
 

In [41]:
missing = df.isnull().sum()
missing[missing > 0]

Series([], dtype: int64)

In [42]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate employee IDs:", df["employee_id"].duplicated().sum())

Duplicate rows: 0
Duplicate employee IDs: 0


In [43]:
df["attrition"].value_counts()

attrition
Stayed    39128
Left      35370
Name: count, dtype: int64

In [44]:
df["attrition"] = df["attrition"].map({"Stayed": 0, "Left": 1})

df["attrition"].value_counts()

attrition
0    39128
1    35370
Name: count, dtype: int64

In [45]:
yes_no_cols = [
    "overtime",
    "remote_work",
    "leadership_opportunities",
    "innovation_opportunities",
]

for col in yes_no_cols:
    df[col] = df[col].map({"Yes": 1, "No": 0})

df[yes_no_cols].head()

,overtime,remote_work,leadership_opportunities,innovation_opportunities
0,0,0,0,0
1,0,0,0,0
2,0,0,0,0
3,0,1,0,0
4,1,0,0,0


In [46]:
numeric_cols = [
    "age",
    "years_at_company",
    "monthly_income",
    "number_of_promotions",
    "distance_from_home",
    "number_of_dependents",
    "company_tenure",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[numeric_cols].describe()

,age,years_at_company,monthly_income,number_of_promotions,distance_from_home,number_of_dependents,company_tenure
count,74498.000000,74498.000000,74498.000000,74498.000000,74498.000000,74498.000000,74498.000000
mean,38.529746,15.721603,7299.379514,0.832935,49.991584,1.650326,55.727456
std,12.083456,11.223744,2152.508566,0.995289,28.513611,1.553633,25.399349
min,18.000000,1.000000,1226.000000,0.000000,1.000000,0.000000,2.000000
25%,28.000000,7.000000,5652.000000,0.000000,25.000000,0.000000,36.000000
50%,39.000000,13.000000,7348.000000,1.000000,50.000000,1.000000,56.000000
75%,49.000000,23.000000,8876.000000,2.000000,75.000000,3.000000,76.000000
max,59.000000,51.000000,16149.000000,4.000000,99.000000,6.000000,128.000000


In [47]:
print("Age range:", df["age"].min(), "to", df["age"].max())
print("Negative income rows:", (df["monthly_income"] < 0).sum())
print("Missing values after cleaning:", df.isnull().sum().sum())

Age range: 18 to 59
Negative income rows: 0
Missing values after cleaning: 0


In [48]:
total_employees = len(df)
left_count = df["attrition"].sum()
attrition_rate = left_count / total_employees * 100

print(f"Total employees: {total_employees:,}")
print(f"Left the company: {left_count:,}")
print(f"Attrition rate: {attrition_rate:.1f}%")

Total employees: 74,498
Left the company: 35,370
Attrition rate: 47.5%


In [49]:
role_attrition = (
    df.groupby("job_role")["attrition"]
    .mean()
    .sort_values(ascending=False)
    * 100
)

role_attrition.round(1)

job_role
Education     48.8
Healthcare    47.5
Technology    47.1
Finance       46.9
Media         46.8
Name: attrition, dtype: float64

In [50]:
gender_attrition = df.groupby("gender")["attrition"].mean() * 100
gender_attrition.round(1)

gender
Female    53.0
Male      42.9
Name: attrition, dtype: float64

In [51]:
income_by_attrition = df.groupby("attrition")["monthly_income"].mean()
income_by_attrition.index = ["Stayed", "Left"]
income_by_attrition.round(0)

Stayed    7321.0
Left      7275.0
Name: monthly_income, dtype: float64

In [52]:
overtime_attrition = df.groupby("overtime")["attrition"].mean() * 100
overtime_attrition.index = ["No Overtime", "Overtime"]
overtime_attrition.round(1)

No Overtime    45.5
Overtime       51.5
Name: attrition, dtype: float64

In [53]:
remote_attrition = df.groupby("remote_work")["attrition"].mean() * 100
remote_attrition.index = ["Not Remote", "Remote"]
remote_attrition.round(1)

Not Remote    52.8
Remote        24.7
Name: attrition, dtype: float64

In [54]:
df.groupby("attrition")[["age", "years_at_company", "distance_from_home"]].mean().round(1)

,age,years_at_company,distance_from_home
attrition,,,
0,39.1,16.4,47.4
1,37.9,14.9,52.8


In [55]:
attrition_labels = df["attrition"].map({0: "Stayed", 1: "Left"})
fig1 = px.pie(
    names=attrition_labels,
    title="Overall Employee Attrition",
    color=attrition_labels,
    color_discrete_map={"Stayed": "#2ecc71", "Left": "#e74c3c"},
)
fig1.show()

In [56]:
role_df = (
    df.groupby("job_role")["attrition"]
    .mean()
    .reset_index()
)
role_df["attrition_pct"] = role_df["attrition"] * 100

fig2 = px.bar(
    role_df.sort_values("attrition_pct", ascending=True),
    x="attrition_pct",
    y="job_role",
    orientation="h",
    title="Attrition Rate by Job Role (%)",
    labels={"attrition_pct": "Attrition Rate (%)", "job_role": "Job Role"},
    color="attrition_pct",
    color_continuous_scale="Reds",
)
fig2.show()

In [57]:
wlb_df = (
    df.groupby("work_life_balance")["attrition"]
    .mean()
    .reset_index()
)
wlb_df["attrition_pct"] = wlb_df["attrition"] * 100

fig3 = px.bar(
    wlb_df.sort_values("attrition_pct"),
    x="work_life_balance",
    y="attrition_pct",
    title="Attrition Rate by Work-Life Balance",
    labels={"attrition_pct": "Attrition Rate (%)", "work_life_balance": "Work-Life Balance"},
    color="attrition_pct",
    color_continuous_scale="Oranges",
)
fig3.show()

In [58]:
ot_df = df.groupby("overtime")["attrition"].mean().reset_index()
ot_df["status"] = ot_df["overtime"].map({0: "No Overtime", 1: "Overtime"})
ot_df["attrition_pct"] = ot_df["attrition"] * 100

fig5 = px.bar(
    ot_df,
    x="status",
    y="attrition_pct",
    title="Attrition Rate: Overtime vs No Overtime",
    labels={"attrition_pct": "Attrition Rate (%)", "status": ""},
    color="status",
    color_discrete_map={"No Overtime": "#3498db", "Overtime": "#e74c3c"},
)
fig5.show()

In [59]:
df.to_csv("cleaned_attrition_data.csv", index=False)
print("Saved cleaned_attrition_data.csv with", len(df), "rows")

Saved cleaned_attrition_data.csv with 74498 rows
